In [ ]:
import h5py
import numpy as np
import faiss
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from scipy.stats import skew

This code is to compare the overlap between 2 Outliers txt list with the jaccard method

In [ ]:
def compare_outliers(file_path_1, file_path_2):
    with open(file_path_1, 'r') as f:
        set1 = {line.strip() for line in f if line.strip()}
    
    with open(file_path_2, 'r') as f:
        set2 = {line.strip() for line in f if line.strip()}

    overlap = set1.intersection(set2)
    
    only_in_1 = set1 - set2
    only_in_2 = set2 - set1

    print(f"--- COMPARAISON DES OUTLIERS ---")
    print(f"Fichier 1 ({file_path_1.split('/')[-1]}) : {len(set1)} IDs")
    print(f"Fichier 2 ({file_path_2.split('/')[-1]}) : {len(set2)} IDs")
    print("-" * 30)
    print(f"Overlap (IDs communs) : {len(overlap)}")
    
    if len(overlap) > 0:
        print(f"Exemples d'overlap : {list(overlap)[:5]}")
        
    if len(set1 | set2) > 0:
        jaccard = len(overlap) / len(set1 | set2) * 100
        print(f"Indice de similitude : {jaccard:.2f}%")

outliers1 = "/home/cassandre/stage/Cassandre/panthr_text/human_mouse/human_mouseoutliers_mus.txt"
outliers2 = "/home/cassandre/stage/Cassandre/UMAP_ESMCEMBEDDINGS/esmc600Moutliers_mus.txt"

compare_outliers(outliers1, outliers2)

--- COMPARAISON DES OUTLIERS ---
Fichier 1 (human_mouseoutliers_mus.txt) : 635 IDs
Fichier 2 (esmc600Moutliers_mus.txt) : 457 IDs
------------------------------
Overlap (IDs communs) : 88
Exemples d'overlap : ['Q8CEK7', 'Q9CWF0', 'S4R2V3', 'E9Q328', 'Q9QYL0']
Indice de similitude : 8.76%


In [1]:
# 1. Define your file paths
file_protT5 = '/home/cassandre/stage/Cassandre/panthr_text/human_mouse/human_mouseoutliers_mus.txt'
file_esm300 = '/home/cassandre/stage/Cassandre/UMAP_ESMCEMBEDDINGS/ESMC300M/outlierslist_text/outliers_mus.txt'
file_esm600 = '/home/cassandre/stage/Cassandre/UMAP_ESMCEMBEDDINGS/esmc600Moutliers_mus.txt'

def load_outliers(filepath):
    with open(filepath, 'r') as f:
        return set(line.strip() for line in f if line.strip())

# 2. Load the sets
A = load_outliers(file_protT5)  # ProtT5
B = load_outliers(file_esm300) # 300M
C = load_outliers(file_esm600) # 600M

# 3. Calculate intersections
only_A = A - (B | C)
only_B = B - (A | C)
only_C = C - (A | B)

A_and_B_only = (A & B) - C
A_and_C_only = (A & C) - B
B_and_C_only = (B & C) - A

all_three = A & B & C

# 4. Print results for your Venn Diagram
print(f"--- Unique to one model ---")
print(f"ProtT5 only: {len(only_A)}")
print(f"ESM 300M only: {len(only_B)}")
print(f"ESM 600M only: {len(only_C)}")

print(f"\n--- Overlaps (Two models only) ---")
print(f"ProtT5 & 300M: {len(A_and_B_only)}")
print(f"ProtT5 & 600M: {len(A_and_C_only)}")
print(f"300M & 600M: {len(B_and_C_only)}")

print(f"\n--- The Center ---")
print(f"Common to all three: {len(all_three)}")

print(f"\n--- Totals (Checking your math) ---")
print(f"Total Unique IDs across all: {len(A | B | C)}")

--- Unique to one model ---
ProtT5 only: 523
ESM 300M only: 172
ESM 600M only: 155

--- Overlaps (Two models only) ---
ProtT5 & 300M: 24
ProtT5 & 600M: 20
300M & 600M: 214

--- The Center ---
Common to all three: 68

--- Totals (Checking your math) ---
Total Unique IDs across all: 1176
